# Stage 3 — Network Analysis

對應範例的 Notebook 2。對 18 個分析單位（14 系所 + 3 組 + 1 全體）執行同一套流程：

1. Professor–Keyword 二分網路與 largest connected component（LCC）
2. Keyword–Keyword 網路（由 LCC 投影，權重 = 共現人數）
3. Professor–Professor 網路（由 LCC 投影，D2 的 IDF + λ 加權）
4. 虛無模型檢定（degree-preserving 二分隨機化 × 1,000）
5. λ 敏感度與 L1 過寬護欄
6. 每單位 17 張圖（組層級與全體再加 3 張）

所有圖檔寫入 `analysis/network-analysis/<unit>/figures/`，`data/` 內不存任何圖檔。
圖中文字一律英文，唯一例外是教授姓名（`professor_name`）。

In [1]:
import json
import math
import random
import re
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
from matplotlib.lines import Line2D
from scipy.stats import pearsonr
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
while not (ROOT / "data" / "data-preprocessing").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

NET_DIR = ROOT / "data" / "network-analysis"
OUT_ROOT = ROOT / "analysis" / "network-analysis"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 42
LAMBDA = 0.4
LAMBDA_GRID = [0.0, 0.3, 0.4, 0.5]
RESOLUTIONS = [0.7, 1.0, 1.3]
N_NULL = 1000
ALPHA_BACKBONE = 0.05
DPI = 180

TAXONOMY = pd.read_csv(NET_DIR / "research_taxonomy.csv", encoding="utf-8-sig")
KW2L2 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l2"]))
KW2L1 = dict(zip(TAXONOMY["keyword_en"], TAXONOMY["parent_l1"]))
IDF_DF = pd.read_csv(NET_DIR / "keyword_idf.csv", encoding="utf-8-sig")
IDF = dict(zip(IDF_DF["keyword_en"], IDF_DF["idf"]))

with (NET_DIR / "units_manifest.json").open(encoding="utf-8") as fh:
    MANIFEST = json.load(fh)

print(f"units: {len(MANIFEST)}   lambda={LAMBDA}   null replicates={N_NULL}")

units: 18   lambda=0.4   null replicates=1000


## 字型偵測（僅用於教授姓名標籤）

依 `msjh.ttc` → `Microsoft JhengHei` → `sans-serif` 順序偵測，不硬寫絕對路徑。
偵測失敗時記錄警告，教授標籤退回 `name_en`。

In [2]:
import matplotlib.font_manager as fm

CJK_RE = re.compile(r"[㐀-䶿一-鿿豈-﫿]")

def detect_cjk_font() -> tuple[str | None, str]:
    for path in fm.findSystemFonts(fontext="ttf") + fm.findSystemFonts(fontext="ttc"):
        if Path(path).name.lower() == "msjh.ttc":
            try:
                name = fm.FontProperties(fname=path).get_name()
                fm.fontManager.addfont(path)
                return name, f"msjh.ttc -> {name}"
            except Exception:
                break
    available = {f.name for f in fm.fontManager.ttflist}
    for cand in ("Microsoft JhengHei", "Microsoft YaHei", "MingLiU", "SimSun",
                 "PingFang TC", "Heiti TC", "Hiragino Sans",
                 "Noto Sans CJK TC", "Noto Sans TC", "Source Han Sans TW",
                 "WenQuanYi Zen Hei", "Droid Sans Fallback"):
        if cand in available:
            return cand, f"font family lookup -> {cand}"
    return None, "no CJK font found; professor labels fall back to name_en"


CJK_FONT, CJK_FONT_NOTE = detect_cjk_font()
FONT_OK = CJK_FONT is not None
if not FONT_OK:
    print(f"WARNING: {CJK_FONT_NOTE}")
print(f"CJK font: {CJK_FONT}  ({CJK_FONT_NOTE})")

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ([CJK_FONT] if FONT_OK else []) + ["DejaVu Sans", "Arial", "sans-serif"],
        "axes.unicode_minus": False,
        "figure.dpi": 110,
        "savefig.dpi": DPI,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
    }
)

# 統一配色（18 個單位共用，圖與圖之間可並排比較）
PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860",
    "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD", "#1F77B4", "#FF7F0E",
    "#2CA02C", "#D62728", "#9467BD", "#8C564B", "#E377C2", "#7F7F7F",
    "#BCBD22", "#17BECF",
]
COLOR_PROF = "#4C72B0"
COLOR_KW = "#DD8452"

L1_ORDER = sorted(TAXONOMY["parent_l1"].unique())
L1_COLOR = {l1: PALETTE[i % len(PALETTE)] for i, l1 in enumerate(L1_ORDER)}
SCHOOL_ORDER = sorted({s for m in MANIFEST if m["level"] == "school" for s in m["school_ids"]})
SCHOOL_COLOR = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(SCHOOL_ORDER)}

def darken(hex_color: str, factor: float = 0.62) -> str:
    """把節點顏色壓暗後給文字用。

    節點配色為了在白底上柔和，明度偏高（例如 `#CCB974`），直接拿來當文字顏色會看不清楚。
    標籤顏色跟著群組走、但壓暗到可讀的明度，才能同時保有「顏色 = 群組」與可讀性。
    """
    hex_color = hex_color.lstrip("#")
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return "#{:02X}{:02X}{:02X}".format(*(max(0, min(255, int(c * factor))) for c in (r, g, b)))


ROLE_PRIORITY = ["full_time", "faculty", "joint", "chair", "adjunct", "visiting", "emeritus", "project", "practice"]

CJK font: Microsoft JhengHei  (msjh.ttc -> Microsoft JhengHei)


## 語言檢查

`save_figure()` 在存檔前掃描圖中**所有文字元素**，凡出現中文字元且不屬於該單位教授姓名者即記錄違規。
這是程式掃描，不是目視檢查。

In [3]:
LANG_CHECK_ROWS: list[dict] = []
FIGURE_INDEX: list[dict] = []


def save_figure(fig, path: Path, unit: str, allowed_cjk: set[str], caption: str | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if caption:
        fig.text(0.005, 0.005, caption, ha="left", va="bottom", fontsize=8, color="#444444")

    texts: list[str] = []
    for ax in fig.get_axes():
        texts.append(ax.get_title())
        texts.append(ax.get_xlabel())
        texts.append(ax.get_ylabel())
        texts += [t.get_text() for t in ax.get_xticklabels()]
        texts += [t.get_text() for t in ax.get_yticklabels()]
        texts += [t.get_text() for t in ax.texts]
        leg = ax.get_legend()
        if leg is not None:
            texts += [t.get_text() for t in leg.get_texts()]
    texts += [t.get_text() for t in fig.texts]
    for leg in getattr(fig, "legends", []):
        texts += [t.get_text() for t in leg.get_texts()]

    violations = []
    for text in texts:
        if not text or not CJK_RE.search(text):
            continue
        if text.strip() in allowed_cjk:
            continue
        violations.append(text.strip())

    LANG_CHECK_ROWS.append(
        {
            "unit": unit,
            "figure": path.name,
            "n_text_elements": len([t for t in texts if t]),
            "n_cjk_elements": len([t for t in texts if t and CJK_RE.search(t)]),
            "n_violations": len(violations),
            "violations": " | ".join(sorted(set(violations))[:5]),
        }
    )
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    FIGURE_INDEX.append({"unit": unit, "figure": path.name, "path": str(path.relative_to(ROOT)).replace("\\", "/")})


def prof_label(node_id: str, attrs: dict) -> str:
    """教授標籤：優先 professor_name（中文優先，缺中文時預處理階段已退回英文）。
    偵測不到中文字型時一律退回 name_en。"""
    name = (attrs.get("professor_name") or "").strip()
    name_en = (attrs.get("name_en") or "").strip()
    if not FONT_OK and CJK_RE.search(name):
        return name_en or node_id
    return name or name_en or node_id

## 網路建構

### D2 — Professor–Professor 邊權重

```
S_exp(A,B) = Σ_{k ∈ K_A ∩ K_B} IDF(k)
S_inf(A,B) = Σ_{p ∈ 共同 L2}  max_{i ∈ K_A\K_B, j ∈ K_B\K_A, L2(i)=L2(j)=p} sqrt(IDF(i)·IDF(j))
w(A,B)     = S_exp + λ·S_inf
w_cos(A,B) = w(A,B) / sqrt( Σ_{K_A} IDF² · Σ_{K_B} IDF² )
```

`S_inf` 每個共同 L2 parent 只取貢獻最大的一組配對。

In [4]:
def build_bipartite(prof_df: pd.DataFrame, edge_df: pd.DataFrame) -> nx.Graph:
    B = nx.Graph()
    for row in prof_df.itertuples(index=False):
        B.add_node(
            row.node_id,
            bipartite=0,
            kind="professor",
            professor_name=str(row.professor_name or ""),
            name_en=str(row.name_en or ""),
            school_ids=str(row.school_ids or ""),
            role_groups=str(row.role_groups or ""),
            n_keywords=int(row.n_keywords),
            phd_institution=str(row.phd_institution) if pd.notna(row.phd_institution) else "",
            n_appointments=int(row.n_appointments),
        )
    for kw in edge_df["keyword_en"].unique():
        B.add_node(
            kw,
            bipartite=1,
            kind="keyword",
            parent_l2=KW2L2[kw],
            parent_l1=KW2L1[kw],
            idf=float(IDF[kw]),
        )
    for row in edge_df.itertuples(index=False):
        B.add_edge(row.node_id, row.keyword_en, source=str(row.source))
    return B


def split_sides(B: nx.Graph) -> tuple[list[str], list[str]]:
    profs = [n for n, d in B.nodes(data=True) if d["bipartite"] == 0]
    kws = [n for n, d in B.nodes(data=True) if d["bipartite"] == 1]
    return sorted(profs), sorted(kws)


def keyword_projection(kw_sets: dict[str, set[str]]) -> nx.Graph:
    """由 professor -> keywords 的對應投影出 keyword–keyword 網路。
    weight = 共同研究該兩領域的教授人數（原始共現次數）；另存 jaccard 與 cosine。"""
    co = Counter()
    kw_count = Counter()
    for kws in kw_sets.values():
        ks = sorted(kws)
        for k in ks:
            kw_count[k] += 1
        for i in range(len(ks)):
            for j in range(i + 1, len(ks)):
                co[(ks[i], ks[j])] += 1
    G = nx.Graph()
    for kw, cnt in kw_count.items():
        G.add_node(
            kw,
            kind="keyword",
            parent_l2=KW2L2[kw],
            parent_l1=KW2L1[kw],
            n_faculty_in_unit=int(cnt),
            idf=float(IDF[kw]),
        )
    for (a, b), w in co.items():
        union = kw_count[a] + kw_count[b] - w
        G.add_edge(
            a,
            b,
            weight=int(w),
            jaccard=w / union if union else 0.0,
            cosine=w / math.sqrt(kw_count[a] * kw_count[b]),
            distance=1.0 / w,
        )
    return G


def professor_projection(kw_sets: dict[str, set[str]], lam: float) -> nx.Graph:
    """D2 公式的加權 professor–professor 投影。"""
    l2_map: dict[str, dict[str, list[str]]] = {}
    norm: dict[str, float] = {}
    for p, kws in kw_sets.items():
        buckets: dict[str, list[str]] = defaultdict(list)
        for k in kws:
            buckets[KW2L2[k]].append(k)
        l2_map[p] = buckets
        norm[p] = math.sqrt(sum(IDF[k] ** 2 for k in kws)) or 1.0

    inv_kw: dict[str, set[str]] = defaultdict(set)
    inv_l2: dict[str, set[str]] = defaultdict(set)
    for p, kws in kw_sets.items():
        for k in kws:
            inv_kw[k].add(p)
        for l2 in l2_map[p]:
            inv_l2[l2].add(p)

    candidates: set[tuple[str, str]] = set()
    for holders in inv_kw.values():
        hs = sorted(holders)
        for i in range(len(hs)):
            for j in range(i + 1, len(hs)):
                candidates.add((hs[i], hs[j]))
    if lam > 0:
        for holders in inv_l2.values():
            hs = sorted(holders)
            for i in range(len(hs)):
                for j in range(i + 1, len(hs)):
                    candidates.add((hs[i], hs[j]))

    G = nx.Graph()
    for p in kw_sets:
        G.add_node(p)
    for a, b in candidates:
        ka, kb = kw_sets[a], kw_sets[b]
        shared = ka & kb
        s_exp = sum(IDF[k] for k in shared)
        s_inf = 0.0
        if lam > 0:
            only_a, only_b = ka - kb, kb - ka
            for l2 in set(l2_map[a]) & set(l2_map[b]):
                best = 0.0
                for i in l2_map[a][l2]:
                    if i not in only_a:
                        continue
                    for j in l2_map[b][l2]:
                        if j not in only_b:
                            continue
                        best = max(best, math.sqrt(IDF[i] * IDF[j]))
                s_inf += best
        w = s_exp + lam * s_inf
        if w <= 0:
            continue
        G.add_edge(
            a,
            b,
            weight=w,
            w_cos=w / (norm[a] * norm[b]),
            s_exp=s_exp,
            s_inf=s_inf,
            n_shared_keywords=len(shared),
            distance=1.0 / w,
        )
    return G

## 中心性、社群偵測與對齊檢驗

In [5]:
def centrality_table(G: nx.Graph, weight: str = "weight") -> pd.DataFrame:
    n = G.number_of_nodes()
    nodes = sorted(G.nodes())
    deg = dict(G.degree())
    strength = dict(G.degree(weight=weight))
    deg_c = nx.degree_centrality(G) if n > 1 else {u: 0.0 for u in nodes}
    if n > 2 and G.number_of_edges() > 0:
        btw = nx.betweenness_centrality(G, weight="distance", normalized=True, seed=SEED)
        clo = nx.closeness_centrality(G, distance="distance")
    else:
        btw = {u: 0.0 for u in nodes}
        clo = {u: 0.0 for u in nodes}
    try:
        eig = nx.eigenvector_centrality_numpy(G, weight=weight) if n > 2 else {u: 0.0 for u in nodes}
    except Exception:
        eig = {u: 0.0 for u in nodes}
    try:
        pr = nx.pagerank(G, weight=weight)
    except Exception:
        pr = {u: 1.0 / max(n, 1) for u in nodes}
    clus = nx.clustering(G, weight=weight) if n > 2 else {u: 0.0 for u in nodes}
    clus_uw = nx.clustering(G) if n > 2 else {u: 0.0 for u in nodes}
    return pd.DataFrame(
        {
            "node": nodes,
            "degree": [deg[u] for u in nodes],
            "degree_centrality": [deg_c[u] for u in nodes],
            "strength": [strength[u] for u in nodes],
            "betweenness": [btw[u] for u in nodes],
            "closeness": [clo[u] for u in nodes],
            "eigenvector": [eig.get(u, 0.0) for u in nodes],
            "pagerank": [pr[u] for u in nodes],
            "clustering_weighted": [clus[u] for u in nodes],
            "clustering": [clus_uw[u] for u in nodes],
        }
    )


def louvain_partition(G: nx.Graph, resolution: float = 1.0) -> tuple[dict[str, int], float, int]:
    if G.number_of_nodes() == 0:
        return {}, float("nan"), 0
    if G.number_of_edges() == 0:
        return {u: i for i, u in enumerate(G.nodes())}, 0.0, G.number_of_nodes()
    comms = nx.community.louvain_communities(G, weight="weight", resolution=resolution, seed=SEED)
    comms = sorted(comms, key=len, reverse=True)
    mapping = {u: i for i, c in enumerate(comms) for u in c}
    q = nx.community.modularity(G, comms, weight="weight", resolution=resolution)
    return mapping, float(q), len(comms)


def girvan_newman_partition(G: nx.Graph, max_levels: int = 12) -> tuple[dict[str, int], float, int]:
    """小圖（< 60 節點）的對照方法：取前 max_levels 層中 modularity 最高者。"""
    if G.number_of_nodes() < 3 or G.number_of_edges() == 0:
        return {}, float("nan"), 0
    best = (None, -1.0, 0)
    gen = nx.community.girvan_newman(G)
    for _ in range(max_levels):
        try:
            comms = next(gen)
        except StopIteration:
            break
        comms = sorted((set(c) for c in comms), key=len, reverse=True)
        q = nx.community.modularity(G, comms, weight="weight")
        if q > best[1]:
            best = ({u: i for i, c in enumerate(comms) for u in c}, float(q), len(comms))
    return best if best[0] is not None else ({}, float("nan"), 0)


def alignment_scores(partition: dict[str, int], labels: dict[str, str]) -> tuple[float, float]:
    common = [u for u in partition if u in labels and labels[u]]
    if len(common) < 2:
        return float("nan"), float("nan")
    a = [partition[u] for u in common]
    b = [labels[u] for u in common]
    if len(set(a)) < 2 and len(set(b)) < 2:
        return 1.0, 1.0
    return float(normalized_mutual_info_score(b, a)), float(adjusted_rand_score(b, a))


def primary_role(role_str: str) -> str:
    roles = [r for r in str(role_str).split(" | ") if r]
    for r in ROLE_PRIORITY:
        if r in roles:
            return r
    return roles[0] if roles else "unknown"

## 虛無模型

保持二分結構與**雙邊 degree 序列**的 double edge swap（`nx.double_edge_swap` 會把
同型節點連起來，破壞二分性，因此改用其二分版本），`nswap = 10 × |E|`，重複 1,000 次。

對比四個觀測量：keyword 網路 density、Louvain modularity Q、平均共現次數、professor 網路 clustering coefficient。

In [6]:
def bipartite_double_edge_swap(edges: np.ndarray, nswap: int, rng: random.Random) -> np.ndarray:
    """(p, k) 邊清單的二分 double edge swap：(p1,k1),(p2,k2) -> (p1,k2),(p2,k1)。
    兩側 degree 序列完全不變，且不產生重邊。"""
    edges = edges.copy()
    m = len(edges)
    present = {(int(p), int(k)) for p, k in edges}
    swaps = 0
    attempts = 0
    max_attempts = nswap * 20
    while swaps < nswap and attempts < max_attempts:
        attempts += 1
        i = rng.randrange(m)
        j = rng.randrange(m)
        if i == j:
            continue
        p1, k1 = int(edges[i, 0]), int(edges[i, 1])
        p2, k2 = int(edges[j, 0]), int(edges[j, 1])
        if p1 == p2 or k1 == k2:
            continue
        if (p1, k2) in present or (p2, k1) in present:
            continue
        present.discard((p1, k1))
        present.discard((p2, k2))
        present.add((p1, k2))
        present.add((p2, k1))
        edges[i, 1] = k2
        edges[j, 1] = k1
        swaps += 1
    return edges


def null_statistics(edges: np.ndarray, n_prof: int, n_kw: int) -> dict[str, float]:
    """由二分邊清單直接以稀疏矩陣算出四個統計量。"""
    rows = edges[:, 0]
    cols = edges[:, 1]
    B = sp.csr_matrix((np.ones(len(edges), dtype=np.float32), (rows, cols)), shape=(n_prof, n_kw))

    C = (B.T @ B).tocoo()
    mask = C.row < C.col
    co_vals = C.data[mask]
    co_rows = C.row[mask]
    co_cols = C.col[mask]
    active = np.unique(np.concatenate([co_rows, co_cols])) if len(co_vals) else np.array([], dtype=int)
    n_kw_active = int(len(active))
    n_edges_kw = int(len(co_vals))
    density_kw = (2 * n_edges_kw / (n_kw_active * (n_kw_active - 1))) if n_kw_active > 1 else 0.0
    mean_co = float(co_vals.mean()) if n_edges_kw else 0.0

    if n_edges_kw:
        Gk = nx.Graph()
        Gk.add_nodes_from(active.tolist())
        Gk.add_weighted_edges_from(zip(co_rows.tolist(), co_cols.tolist(), co_vals.tolist()))
        comms = nx.community.louvain_communities(Gk, weight="weight", resolution=1.0, seed=SEED)
        q_kw = float(nx.community.modularity(Gk, comms, weight="weight"))
    else:
        q_kw = 0.0

    P = (B @ B.T).toarray()
    np.fill_diagonal(P, 0)
    A = (P > 0).astype(np.float32)
    deg = A.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        tri = np.einsum("ij,ij->i", A @ A, A) / 2.0
        denom = deg * (deg - 1) / 2.0
        cc = np.where(denom > 0, tri / np.maximum(denom, 1e-12), 0.0)
    clustering_prof = float(cc.mean()) if len(cc) else 0.0

    return {
        "keyword_density": density_kw,
        "keyword_modularity_q": q_kw,
        "keyword_mean_cooccurrence": mean_co,
        "professor_clustering": clustering_prof,
    }


def run_null_model(kw_sets: dict[str, set[str]], n_rep: int = N_NULL):
    profs = sorted(kw_sets)
    kws = sorted({k for s in kw_sets.values() for k in s})
    pidx = {p: i for i, p in enumerate(profs)}
    kidx = {k: i for i, k in enumerate(kws)}
    edges = np.array([(pidx[p], kidx[k]) for p in profs for k in sorted(kw_sets[p])], dtype=np.int32)

    observed = null_statistics(edges, len(profs), len(kws))
    nswap = 10 * len(edges)
    rng = random.Random(SEED)
    samples = defaultdict(list)
    for _ in range(n_rep):
        rand_edges = bipartite_double_edge_swap(edges, nswap, rng)
        stats = null_statistics(rand_edges, len(profs), len(kws))
        for k, v in stats.items():
            samples[k].append(v)

    rows = []
    for metric, obs in observed.items():
        arr = np.array(samples[metric], dtype=float)
        mu, sd = float(arr.mean()), float(arr.std(ddof=1))
        z = (obs - mu) / sd if sd > 0 else float("nan")
        p_greater = float((np.sum(arr >= obs) + 1) / (len(arr) + 1))
        p_less = float((np.sum(arr <= obs) + 1) / (len(arr) + 1))
        rows.append(
            {
                "metric": metric,
                "observed": obs,
                "null_mean": mu,
                "null_std": sd,
                "null_p05": float(np.percentile(arr, 5)),
                "null_p95": float(np.percentile(arr, 95)),
                "z_score": z,
                "p_empirical_two_sided": float(min(1.0, 2 * min(p_greater, p_less))),
                "n_replicates": len(arr),
            }
        )
    return pd.DataFrame(rows), {k: np.array(v) for k, v in samples.items()}, observed

## 繪圖工具

- **指標一律以完整加權圖計算；backbone 抽取（disparity filter, α=0.05）只用於繪圖。**
- 統一 `spring_layout(seed=42)` 與統一配色。
- 節點數 > 80 時只標註 top-15 標籤。
- 圖檔尺寸上限 20×16 吋、`dpi=180`。

In [7]:
BACKBONE_MIN_EDGES = 400


def disparity_backbone(G: nx.Graph, alpha: float = ALPHA_BACKBONE) -> nx.Graph:
    """Serrano et al. (2009) disparity filter，並保證每個節點至少保留其最強的一條邊
    （否則低度數節點會被整批切掉，圖變成一堆孤立點）。"""
    keep: set[tuple] = set()
    strength = dict(G.degree(weight="weight"))
    for u in G.nodes():
        k = G.degree(u)
        if k == 0:
            continue
        if k == 1:
            keep.update((u, v) for v in G.neighbors(u))
            continue
        s = strength[u] or 1.0
        for v, data in G[u].items():
            pij = data["weight"] / s
            if (1 - pij) ** (k - 1) < alpha:
                keep.add((u, v))
        best = max(G[u].items(), key=lambda kv: kv[1]["weight"])[0]
        keep.add((u, best))
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    for u, v in keep:
        H.add_edge(u, v, **G[u][v])
    return H


def backbone_for_plot(G: nx.Graph, alpha: float = ALPHA_BACKBONE) -> tuple[nx.Graph, bool]:
    """只有在完整圖已經密到畫不出結構時（邊數 > 400）才抽 backbone；否則直接畫完整圖。
    節點座標一律由**完整加權圖**計算，backbone 只影響畫出來的邊。
    回傳 (圖, 是否套用 backbone)，caption 依此標註。"""
    if G.number_of_edges() <= BACKBONE_MIN_EDGES:
        return G.copy(), False
    H = disparity_backbone(G, alpha)
    if H.number_of_edges() == 0:
        return G.copy(), False
    return H, True


def scaled_sizes(values: dict, nodes: list, lo: float = 40, hi: float = 900) -> list[float]:
    vals = np.array([values.get(n, 0.0) for n in nodes], dtype=float)
    if vals.max() - vals.min() < 1e-12:
        return [(lo + hi) / 2] * len(nodes)
    norm = (vals - vals.min()) / (vals.max() - vals.min())
    return list(lo + norm * (hi - lo))


def edge_widths(G: nx.Graph, nodes_order=None, lo: float = 0.2, hi: float = 3.0) -> list[float]:
    w = np.array([d.get("weight", 1.0) for _, _, d in G.edges(data=True)], dtype=float)
    if len(w) == 0:
        return []
    if w.max() - w.min() < 1e-12:
        return [(lo + hi) / 2] * len(w)
    return list(lo + (w - w.min()) / (w.max() - w.min()) * (hi - lo))


def top_label_set(values: dict, nodes: list, limit: int = 15) -> set:
    if len(nodes) <= 80:
        return set(nodes)
    return set(sorted(nodes, key=lambda n: values.get(n, 0.0), reverse=True)[:limit])


def caption_for(G: nx.Graph, backbone: bool, lam: float | None) -> str:
    parts = [f"nodes = {G.number_of_nodes()}", f"edges = {G.number_of_edges()}"]
    parts.append(
        f"backbone = disparity filter alpha={ALPHA_BACKBONE}" if backbone else "backbone = none (full graph shown)"
    )
    parts.append(f"lambda = {lam:.2f}" if lam is not None else "lambda = n/a (co-occurrence weights)")
    return "  |  ".join(parts)


def _marker_radius_px(node_size_pts2: float, fig) -> float:
    """matplotlib 的 node_size 是點數平方（面積），換算成顯示座標的半徑（像素）。"""
    return math.sqrt(max(node_size_pts2, 1.0) / math.pi) * fig.dpi / 72.0


def annotate_without_overlap(
    ax,
    pos: dict,
    label_map: dict,
    size_map: dict,
    font_size: float = 8.0,
    obstacle_nodes: list | None = None,
    label_colors: dict | None = None,
    iterations: int = 150,
    pad: float = 1.5,
) -> None:
    """把節點標籤排開，避免文字互相疊在一起。

    做法（不依賴外部套件，`adjustText` 未安裝）：

    1. 先把每個標籤放在節點正上方（偏移量 = 節點半徑 + 半個字高）。
    2. 在**顯示座標（像素）**上反覆檢查標籤矩形兩兩是否重疊，沿重疊較淺的軸把兩者推開；
       同時把標籤推出所有節點的圓形標記之外。標籤只有在發生碰撞時才移動，因此位移量天然最小。
    3. 位移超過門檻的標籤畫一條細引線回到自己的節點，讀者才知道標籤屬於誰。

    標籤加白色半透明底框，壓在邊線之上仍可辨識。`label_colors` 可指定每個節點的文字顏色
    （用於依社群／分類上色的圖），未指定者為深灰。引線顏色亦跟隨標籤顏色。
    """
    items = [(n, str(t)) for n, t in label_map.items() if t and n in pos]
    if not items:
        return
    fig = ax.figure

    # 先把座標軸範圍撐開，標籤才有地方可以移動
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    mx, my = (x1 - x0) * 0.08, (y1 - y0) * 0.08
    ax.set_xlim(x0 - mx, x1 + mx)
    ax.set_ylim(y0 - my, y1 + my)
    ax.autoscale(False)

    fig.canvas.draw()
    trans = ax.transData

    texts, anchors_px, radii_px = [], [], []
    for n, label in items:
        r = _marker_radius_px(size_map.get(n, 60.0), fig)
        ax_px = trans.transform(pos[n])
        anchors_px.append(ax_px)
        radii_px.append(r)
        start = trans.inverted().transform(ax_px + np.array([0.0, r + font_size * 0.75]))
        texts.append(
            ax.text(
                start[0], start[1], label,
                fontsize=font_size, ha="center", va="center", zorder=6,
                color=(label_colors or {}).get(n, "#111111"),
                bbox=dict(boxstyle="round,pad=0.12", facecolor="white", edgecolor="none", alpha=0.7),
            )
        )

    obstacles = obstacle_nodes if obstacle_nodes is not None else list(pos)
    obs_xy = np.array([trans.transform(pos[n]) for n in obstacles if n in pos], dtype=float)
    obs_r = np.array([_marker_radius_px(size_map.get(n, 60.0), fig) for n in obstacles if n in pos], dtype=float)

    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    boxes = [t.get_window_extent(rend) for t in texts]
    w = np.array([b.width + 2 * pad for b in boxes], dtype=float)
    h = np.array([b.height + 2 * pad for b in boxes], dtype=float)
    cur = np.array([[(b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2] for b in boxes], dtype=float)
    anchors_px = np.asarray(anchors_px, dtype=float)
    radii_px = np.asarray(radii_px, dtype=float)
    n_lab = len(texts)

    for _ in range(iterations):
        moved = False
        for i in range(n_lab):
            for j in range(i + 1, n_lab):
                dx = cur[j, 0] - cur[i, 0]
                dy = cur[j, 1] - cur[i, 1]
                ox = (w[i] + w[j]) / 2 - abs(dx)
                oy = (h[i] + h[j]) / 2 - abs(dy)
                if ox <= 0 or oy <= 0:
                    continue
                moved = True
                if oy <= ox:
                    step = oy / 2 + 0.5
                    sgn = 1.0 if dy >= 0 else -1.0
                    cur[i, 1] -= sgn * step
                    cur[j, 1] += sgn * step
                else:
                    step = ox / 2 + 0.5
                    sgn = 1.0 if dx >= 0 else -1.0
                    cur[i, 0] -= sgn * step
                    cur[j, 0] += sgn * step
        if len(obs_xy):
            for i in range(n_lab):
                d = cur[i] - obs_xy
                hit = (np.abs(d[:, 0]) < w[i] / 2 + obs_r) & (np.abs(d[:, 1]) < h[i] / 2 + obs_r)
                if not hit.any():
                    continue
                moved = True
                push = d[hit]
                norm = np.hypot(push[:, 0], push[:, 1])
                norm[norm < 1e-6] = 1e-6
                vec = (push / norm[:, None]).sum(axis=0)
                vlen = math.hypot(vec[0], vec[1]) or 1.0
                cur[i] += (vec / vlen) * 2.5
        if not moved:
            break

    inv = trans.inverted()
    for t, target in zip(texts, cur):
        t.set_position(inv.transform(target))
    for (node, _), target, a, r in zip(items, cur, anchors_px, radii_px):
        if math.hypot(target[0] - a[0], target[1] - a[1]) > r + font_size * 1.6:
            (lx0, ly0) = inv.transform(a)
            (lx1, ly1) = inv.transform(target)
            ax.plot([lx0, lx1], [ly0, ly1], lw=0.4, zorder=3, solid_capstyle="round",
                    color=(label_colors or {}).get(node, "#B8BEC7"), alpha=0.55)


def draw_network(
    ax,
    G: nx.Graph,
    pos: dict,
    node_colors,
    size_values: dict,
    labels_for: set,
    label_fn,
    font_size: int = 8,
    edge_alpha: float = 0.25,
    label_colors: dict | None = None,
) -> None:
    nodes = list(G.nodes())
    n = len(nodes)
    lo, hi = (40, 900) if n <= 60 else (30, 520) if n <= 150 else (18, 300)
    sizes = scaled_sizes(size_values, nodes, lo, hi)
    nx.draw_networkx_edges(ax=ax, G=G, pos=pos, width=edge_widths(G), alpha=edge_alpha, edge_color="#999999")
    nx.draw_networkx_nodes(
        ax=ax,
        G=G,
        pos=pos,
        nodelist=nodes,
        node_size=sizes,
        node_color=node_colors,
        linewidths=0.4,
        edgecolors="white",
    )
    ax.set_axis_off()
    annotate_without_overlap(
        ax,
        pos,
        {u: label_fn(u) for u in nodes if u in labels_for},
        dict(zip(nodes, sizes)),
        font_size=font_size,
        obstacle_nodes=nodes,
        label_colors=label_colors,
    )

## Word cloud 排版

`wordcloud` 套件無法把同一個社群的詞放在一起，中文也需要另外指定字型檔路徑，
因此自行排版：以 `TextPath` 量測字串實際外框、用佔用點陣圖（occupancy bitmap）做碰撞偵測、
沿阿基米德螺線由中心往外找空位。

- **字級與中心性成正比**：`fontsize = FS_MAX × value / max(value)`，並設下限 `FS_MIN`
  （betweenness 有大量 0 值，沒有下限會整個消失）。
- **社群版本**：以 squarified treemap 把畫布切成數塊，每塊面積正比於該社群的字面積總和，
  同社群的詞只在自己的區塊內排版並共用顏色。

In [8]:
WC_FS_MAX = 46.0
WC_FS_MIN = 7.0
WC_MAX_WORDS = 150
WC_GRID = 1.0                 # 佔用點陣圖解析度：1 格 = 1 點
WC_ASPECT = 1.45              # 畫布長寬比
WC_FILL = 0.30                # 目標填充率：字面積 / 畫布面積
WC_CAPTION_BAND = 30.0        # 底部保留給 caption 的空白（點）
WC_REF_SIZE = 100.0
_WC_MEASURE_CACHE: dict[str, tuple[float, float, float, float]] = {}


def _wc_measure(ax, text: str) -> tuple[float, float, float, float]:
    """在參考字級量一次字串的**排版框**，回傳 (w, h, dx, dy)，單位一律為點。

    `dx` / `dy` 是從 `ha="left", va="baseline"` 錨點到排版框左下角的位移。

    不用 `TextPath`：那給的是字形**墨水框**，實際算繪用的是**排版框**
    （advance width × line height）。兩者對英文只差 1%，但**中文差到 21%**，
    拿墨水框配置空間就會出現字疊在一起。改用真正的 renderer 量一次再依字級
    線性縮放（同一字串的排版框與字級成正比），並快取；換算成點之後與圖檔尺寸無關。
    """
    hit = _WC_MEASURE_CACHE.get(text)
    if hit is not None:
        return hit
    fig = ax.figure
    px2pt = 72.0 / fig.dpi
    probe = ax.text(0, 0, text, fontsize=WC_REF_SIZE, ha="left", va="baseline")
    e = probe.get_window_extent(fig.canvas.get_renderer())
    ax0, ay0 = ax.transData.transform((0.0, 0.0))
    hit = ((e.x1 - e.x0) * px2pt, (e.y1 - e.y0) * px2pt,
           (e.x0 - ax0) * px2pt, (e.y0 - ay0) * px2pt)
    probe.remove()
    _WC_MEASURE_CACHE[text] = hit
    return hit


def _wc_extent(ax, text: str, fs: float) -> tuple[float, float, float, float]:
    """指定字級下的排版框（點）：(寬, 高, dx, dy)，寬高含少量留白。"""
    w, h, dx, dy = _wc_measure(ax, text)
    k = fs / WC_REF_SIZE
    return w * k + fs * 0.14, h * k + fs * 0.10, dx * k, dy * k


def _squarify(values: list[float], x: float, y: float, w: float, h: float) -> list[tuple]:
    """Squarified treemap：回傳與 values 等長、面積正比於 values 的矩形清單。"""
    order = sorted(range(len(values)), key=lambda i: values[i], reverse=True)
    total = sum(values) or 1.0
    scaled = [values[i] * w * h / total for i in order]
    rects: list[tuple] = []
    cx, cy, cw, ch = x, y, w, h
    row: list[float] = []

    def worst(rw: list[float], side: float) -> float:
        s = sum(rw)
        if s <= 0 or side <= 0:
            return float("inf")
        return max(side * side * max(rw) / (s * s), s * s / (side * side * min(rw)))

    def flush(rw, cx, cy, cw, ch):
        out, s = [], sum(rw)
        if cw >= ch:
            rw_width = s / ch if ch else 0.0
            yy = cy
            for v in rw:
                hh = v / rw_width if rw_width else 0.0
                out.append((cx, yy, cx + rw_width, yy + hh))
                yy += hh
            return out, (cx + rw_width, cy, cw - rw_width, ch)
        rh = s / cw if cw else 0.0
        xx = cx
        for v in rw:
            ww = v / rh if rh else 0.0
            out.append((xx, cy, xx + ww, cy + rh))
            xx += ww
        return out, (cx, cy + rh, cw, ch - rh)

    for v in scaled:
        side = min(cw, ch)
        if row and worst(row + [v], side) > worst(row, side):
            placed, (cx, cy, cw, ch) = flush(row, cx, cy, cw, ch)
            rects.extend(placed)
            row = [v]
        else:
            row.append(v)
    if row:
        placed, _ = flush(row, cx, cy, cw, ch)
        rects.extend(placed)

    out = [None] * len(values)
    for slot, i in enumerate(order):
        out[i] = rects[slot] if slot < len(rects) else (x, y, x + w, y + h)
    return out


class _Occupancy:
    """畫布佔用點陣圖；矩形是否可放只需一次 numpy 子陣列查詢。"""

    def __init__(self, w: float, h: float, cell: float = WC_GRID) -> None:
        self.cell = cell
        self.nx = int(math.ceil(w / cell)) + 2
        self.ny = int(math.ceil(h / cell)) + 2
        self.grid = np.zeros((self.ny, self.nx), dtype=bool)

    def _idx(self, x0, y0, x1, y1):
        return (max(0, int(x0 / self.cell)), max(0, int(y0 / self.cell)),
                min(self.nx, int(math.ceil(x1 / self.cell))),
                min(self.ny, int(math.ceil(y1 / self.cell))))

    def free(self, x0, y0, x1, y1) -> bool:
        gx0, gy0, gx1, gy1 = self._idx(x0, y0, x1, y1)
        if gx1 <= gx0 or gy1 <= gy0:
            return False
        return not self.grid[gy0:gy1, gx0:gx1].any()

    def mark(self, x0, y0, x1, y1) -> None:
        gx0, gy0, gx1, gy1 = self._idx(x0, y0, x1, y1)
        self.grid[gy0:gy1, gx0:gx1] = True


def _spiral_place(items, rect, occ: "_Occupancy", max_steps: int = 3000):
    """在 rect 內沿阿基米德螺線由中心往外找空位。"""
    x0, y0, x1, y1 = rect
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    span = max(x1 - x0, y1 - y0)
    aspect = (x1 - x0) / max(y1 - y0, 1e-6)
    results = []
    for it in items:
        w, h = it["w"], it["h"]
        if w > (x1 - x0) or h > (y1 - y0):
            results.append((it, None, None))
            continue
        theta, placed = 0.0, False
        for _ in range(max_steps):
            r = 0.9 * theta
            px = cx + r * aspect * math.cos(theta)
            py = cy + r * math.sin(theta)
            theta += 0.32
            rx0, ry0, rx1, ry1 = px - w / 2, py - h / 2, px + w / 2, py + h / 2
            if rx0 < x0 or rx1 > x1 or ry0 < y0 or ry1 > y1:
                if r > 2.2 * span:
                    break
                continue
            if occ.free(rx0, ry0, rx1, ry1):
                occ.mark(rx0, ry0, rx1, ry1)
                it["rect"] = (rx0, ry0, rx1, ry1)
                results.append((it, px, py))
                placed = True
                break
        if not placed:
            results.append((it, None, None))
    return results


def draw_wordcloud(ax, entries: list[dict], cluster_of: dict | None = None,
                   cluster_colors: dict | None = None) -> dict:
    """畫 word cloud，字級與 `value` 成正比。

    `entries` 每筆為 `{"key", "text", "value", "color"}`。
    給定 `cluster_of` 時，畫布以 squarified treemap 依社群切塊，同社群的詞排在同一塊、同顏色。

    **座標系**：`fontsize` 的單位是點，而排版座標是資料單位；若兩者比例不是 1，
    保留的空間就會與實際字形大小對不上（預設 axes 只佔畫布約 77%，字會疊在一起）。
    因此這裡把 axes 撐滿整張圖並令 `xlim/ylim` 等於畫布點數，強制 **1 資料單位 = 1 點**，
    且排版完成後不再改變圖檔尺寸或座標範圍。
    """
    fig = ax.figure
    entries = [e for e in entries if str(e.get("text", "")).strip()]
    entries = sorted(entries, key=lambda e: -e["value"])[:WC_MAX_WORDS]
    if not entries:
        ax.set_axis_off()
        return {"n_words": 0, "n_dropped": 0, "fs_max": 0.0, "fs_min": 0.0}

    vmax = max(e["value"] for e in entries) or 1.0
    scale, fill, drawn, cells_drawn, dropped = 1.0, WC_FILL, [], [], len(entries)
    at_max_canvas = False

    for _ in range(8):
        fig.canvas.draw()
        fs_max, fs_min = WC_FS_MAX * scale, max(4.0, WC_FS_MIN * scale)
        items = []
        for e in entries:
            fs = max(fs_min, fs_max * e["value"] / vmax)
            w, h, dx, dy = _wc_extent(ax, str(e["text"]), fs)
            items.append({**e, "fs": fs, "w": w, "h": h, "ox": dx, "oy": dy})

        # 畫布大小由字面積推得，避免大片空白
        area = sum(i["w"] * i["h"] for i in items) / fill
        W_raw = math.sqrt(area * WC_ASPECT)
        W = min(max(W_raw, 420.0), 20 * 72.0)
        H = min(max(W / WC_ASPECT, 300.0), 16 * 72.0)
        at_max_canvas = W >= 20 * 72.0 - 1 or H >= 16 * 72.0 - 1
        fig.set_size_inches(W / 72, H / 72)
        ax.set_position([0.0, 0.0, 1.0, 1.0])
        ax.set_xlim(0, W)
        ax.set_ylim(0, H)
        ax.set_aspect("equal")

        occ = _Occupancy(W, H)
        region = (0.0, WC_CAPTION_BAND, W, H)
        cells_drawn = []
        if cluster_of is None:
            placed = _spiral_place(items, region, occ)
        else:
            groups: dict = {}
            for it in items:
                groups.setdefault(cluster_of.get(it["key"], -1), []).append(it)
            keys = sorted(groups, key=lambda c: -sum(i["w"] * i["h"] for i in groups[c]))
            # 面積正比於該社群的字面積，但保底到「最大的那個詞的 4 倍」，
            # 否則只有兩三個人的小社群會被切成細長條，字根本放不進去
            areas = [max(sum(i["w"] * i["h"] for i in groups[c]),
                         4.0 * max(i["w"] * i["h"] for i in groups[c])) for c in keys]
            # squarify 只保證面積，不保證形狀：小社群仍可能被切成放不下字的細長條。
            # 逐一檢查每個格子是否容得下該社群最寬／最高的詞，不夠就加大它的面積重切。
            need_w = [max(i["w"] for i in groups[c]) + 2 * 6.0 for c in keys]
            need_h = [max(i["h"] for i in groups[c]) + 2 * 6.0 + 15.0 for c in keys]
            for _ in range(6):
                cells = _squarify(areas, region[0], region[1],
                                  region[2] - region[0], region[3] - region[1])
                tight = [k for k, cell in enumerate(cells)
                         if (cell[2] - cell[0]) < need_w[k] or (cell[3] - cell[1]) < need_h[k]]
                if not tight:
                    break
                for k in tight:
                    areas[k] *= 1.7
            placed = []
            for c, cell in zip(keys, cells):
                pad = 6.0
                inner = (cell[0] + pad, cell[1] + pad, cell[2] - pad, cell[3] - pad - 15.0)
                placed += _spiral_place(groups[c], inner, occ)
                cells_drawn.append((c, cell, len(groups[c])))

        dropped = sum(1 for _, x, _ in placed if x is None)
        drawn = placed
        if dropped <= max(1, int(0.03 * len(items))):
            break
        # 放不下時**優先放大畫布**而不是縮小字：縮字會一路撞到字級下限，
        # 既犧牲「字級與中心性成正比」的可讀性，也未必真的放得下。
        # 只有畫布已達 20×16 吋上限時才退而縮小字級。
        if at_max_canvas:
            scale *= 0.85
        else:
            fill *= 0.78

    for c, cell, n in cells_drawn:
        col = (cluster_colors or {}).get(c, "#DDDDDD")
        ax.add_patch(matplotlib.patches.Rectangle(
            (cell[0], cell[1]), cell[2] - cell[0], cell[3] - cell[1],
            facecolor=col, alpha=0.07, edgecolor=col, lw=0.8, zorder=0))
        ax.text(cell[0] + 7, cell[3] - 7, f"Community {c}  (n = {n})",
                fontsize=9, ha="left", va="top", zorder=2, color=darken(col))

    for it, px, py in drawn:
        if px is None:
            continue
        rx0, ry0, _, _ = it["rect"]
        ax.text(rx0 + it["fs"] * 0.07 - it["ox"], ry0 + it["fs"] * 0.05 - it["oy"],
                str(it["text"]), fontsize=it["fs"], ha="left", va="baseline",
                color=it["color"], zorder=3)

    ax.set_axis_off()
    return {"n_words": len(drawn), "n_dropped": dropped,
            "fs_max": WC_FS_MAX * scale, "fs_min": max(4.0, WC_FS_MIN * scale)}


def wordcloud_caption(stats: dict, metric: str, n_total: int, lam: float | None = None) -> str:
    parts = [
        f"font size proportional to {metric} (floor = {stats['fs_min']:.1f} pt)",
        f"words shown = {stats['n_words'] - stats['n_dropped']} of {n_total}",
    ]
    if stats["n_dropped"]:
        parts.append(f"{stats['n_dropped']} omitted (no space)")
    if n_total > WC_MAX_WORDS:
        parts.append(f"top {WC_MAX_WORDS} by {metric}")
    if lam is not None:
        parts.append(f"lambda = {lam:.2f}")
    return "  |  ".join(parts)

## 單一分析單位的完整流程

In [9]:
def analyse_unit(meta: dict, n_null: int = N_NULL) -> dict:
    unit = meta["unit"]
    level = meta["level"]
    in_dir = ROOT / meta["out_dir"]
    out_dir = OUT_ROOT / unit
    fig_dir = out_dir / "figures"
    out_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    prof_df = pd.read_csv(in_dir / "professor_nodes.csv", encoding="utf-8-sig")
    kw_df = pd.read_csv(in_dir / "keyword_nodes.csv", encoding="utf-8-sig")
    edge_df = pd.read_csv(in_dir / "prof_keyword_edges.csv", encoding="utf-8-sig")

    B = build_bipartite(prof_df, edge_df)
    profs_all, kws_all = split_sides(B)
    ALLOWED_CJK = {str(v).strip() for v in prof_df["professor_name"].dropna()} | {
        str(v).strip() for v in prof_df["name_en"].dropna()
    }

    # ---- 3.1 二分網路層級指標 -------------------------------------------
    n_components = nx.number_connected_components(B)
    lcc_nodes = max(nx.connected_components(B), key=len) if B.number_of_nodes() else set()
    LCC = B.subgraph(lcc_nodes).copy()
    lcc_profs = sorted(n for n in LCC if LCC.nodes[n]["bipartite"] == 0)
    lcc_kws = sorted(n for n in LCC if LCC.nodes[n]["bipartite"] == 1)

    prof_deg = dict(B.degree(profs_all))
    kw_deg = dict(B.degree(kws_all))
    bip_density = nx.bipartite.density(B, profs_all) if profs_all and kws_all else 0.0

    kw_sets = {p: set(LCC.neighbors(p)) for p in lcc_profs}

    # ---- 3.2 Keyword–Keyword ------------------------------------------
    KG = keyword_projection(kw_sets)
    kw_cent = centrality_table(KG)
    kw_cent["parent_l1"] = kw_cent["node"].map(KW2L1)
    kw_cent["parent_l2"] = kw_cent["node"].map(KW2L2)
    kw_cent["n_faculty_in_unit"] = kw_cent["node"].map(lambda k: KG.nodes[k]["n_faculty_in_unit"])
    kw_cent = kw_cent.sort_values("degree_centrality", ascending=False).reset_index(drop=True)

    kw_parts: dict[float, dict] = {}
    kw_q: dict[float, float] = {}
    kw_ncomm: dict[float, int] = {}
    for res in RESOLUTIONS:
        part, q, nc = louvain_partition(KG, res)
        kw_parts[res], kw_q[res], kw_ncomm[res] = part, q, nc
    kw_part = kw_parts[1.0]

    kw_gn = {}
    if KG.number_of_nodes() < 60:
        gn_part, gn_q, gn_nc = girvan_newman_partition(KG)
        kw_gn = {"n_communities": gn_nc, "modularity_q": gn_q}

    l1_labels = {k: KW2L1[k] for k in KG.nodes()}
    nmi_kw, ari_kw = alignment_scores(kw_part, l1_labels)

    if len(kw_cent) > 2:
        r_deg_btw, p_deg_btw = pearsonr(kw_cent["degree"], kw_cent["betweenness"])
    else:
        r_deg_btw, p_deg_btw = float("nan"), float("nan")

    kw_comm_df = pd.DataFrame(
        {
            "node": sorted(KG.nodes()),
            "parent_l1": [KW2L1[k] for k in sorted(KG.nodes())],
            "parent_l2": [KW2L2[k] for k in sorted(KG.nodes())],
            **{f"louvain_res_{res}": [kw_parts[res].get(k, -1) for k in sorted(KG.nodes())] for res in RESOLUTIONS},
        }
    )
    kw_comm_df["community"] = kw_comm_df["louvain_res_1.0"]

    # ---- 3.3 Professor–Professor --------------------------------------
    PG = professor_projection(kw_sets, LAMBDA)
    for n in PG.nodes():
        PG.nodes[n].update(B.nodes[n])
    prof_cent = centrality_table(PG)
    prof_cent["professor_name"] = prof_cent["node"].map(lambda n: B.nodes[n]["professor_name"])
    prof_cent["name_en"] = prof_cent["node"].map(lambda n: B.nodes[n]["name_en"])
    prof_cent["school_ids"] = prof_cent["node"].map(lambda n: B.nodes[n]["school_ids"])
    prof_cent["n_appointments"] = prof_cent["node"].map(lambda n: B.nodes[n]["n_appointments"])

    dominant_l1 = {}
    for p, kws in kw_sets.items():
        c = Counter(KW2L1[k] for k in kws)
        best = max(c.items(), key=lambda kv: (kv[1], -sorted(c).index(kv[0])))
        dominant_l1[p] = best[0]
    prof_cent["dominant_l1"] = prof_cent["node"].map(dominant_l1)
    prof_cent["primary_role"] = prof_cent["node"].map(lambda n: primary_role(B.nodes[n]["role_groups"]))
    prof_cent = prof_cent.sort_values("degree_centrality", ascending=False).reset_index(drop=True)

    for n in PG.nodes():
        PG.nodes[n]["dominant_l1"] = dominant_l1[n]
        PG.nodes[n]["primary_role"] = primary_role(B.nodes[n]["role_groups"])

    prof_parts, prof_q, prof_ncomm = {}, {}, {}
    for res in RESOLUTIONS:
        part, q, nc = louvain_partition(PG, res)
        prof_parts[res], prof_q[res], prof_ncomm[res] = part, q, nc
    prof_part = prof_parts[1.0]

    prof_comm_df = pd.DataFrame(
        {
            "node": sorted(PG.nodes()),
            "professor_name": [B.nodes[n]["professor_name"] for n in sorted(PG.nodes())],
            "name_en": [B.nodes[n]["name_en"] for n in sorted(PG.nodes())],
            "school_ids": [B.nodes[n]["school_ids"] for n in sorted(PG.nodes())],
            "dominant_l1": [dominant_l1[n] for n in sorted(PG.nodes())],
            **{f"louvain_res_{res}": [prof_parts[res].get(n, -1) for n in sorted(PG.nodes())] for res in RESOLUTIONS},
        }
    )
    prof_comm_df["community"] = prof_comm_df["louvain_res_1.0"]

    try:
        assort_l1 = float(nx.attribute_assortativity_coefficient(PG, "dominant_l1"))
    except Exception:
        assort_l1 = float("nan")
    try:
        assort_role = float(nx.attribute_assortativity_coefficient(PG, "primary_role"))
    except Exception:
        assort_role = float("nan")
    assort_school = float("nan")
    if level != "school":
        for n in PG.nodes():
            PG.nodes[n]["school_primary"] = str(B.nodes[n]["school_ids"]).split(" | ")[0]
        try:
            assort_school = float(nx.attribute_assortativity_coefficient(PG, "school_primary"))
        except Exception:
            assort_school = float("nan")

    nmi_prof_l1, ari_prof_l1 = alignment_scores(prof_part, dominant_l1)

    # ---- 3.5 λ 敏感度 ---------------------------------------------------
    lam_rows = []
    base_density = None
    base_q = None
    for lam in LAMBDA_GRID:
        Gl = PG if abs(lam - LAMBDA) < 1e-9 else professor_projection(kw_sets, lam)
        n, m = Gl.number_of_nodes(), Gl.number_of_edges()
        dens = nx.density(Gl)
        mean_deg = 2 * m / n if n else 0.0
        lcc_share = (len(max(nx.connected_components(Gl), key=len)) / n) if n else 0.0
        _, q, nc = louvain_partition(Gl, 1.0)
        if lam == 0.0:
            base_density, base_q = dens, q
        lam_rows.append(
            {
                "unit": unit,
                "lambda": lam,
                "n_nodes": n,
                "n_edges": m,
                "density": dens,
                "mean_degree": mean_deg,
                "lcc_share": lcc_share,
                "n_communities": nc,
                "modularity_q": q,
            }
        )
    lam_df = pd.DataFrame(lam_rows)
    lam_df["density_change_vs_baseline"] = lam_df["density"] / base_density - 1 if base_density else float("nan")
    lam_df["modularity_drop_vs_baseline"] = base_q - lam_df["modularity_q"]
    lam_df["flag_over_merging"] = (lam_df["density_change_vs_baseline"] > 0.5) | (
        lam_df["modularity_drop_vs_baseline"] > 0.1
    )

    # L1 過寬護欄
    l1_cov_rows = []
    n_unit_profs = len(kw_sets)
    for l1 in sorted({KW2L1[k] for k in KG.nodes()}):
        holders = sum(1 for kws in kw_sets.values() if any(KW2L1[k] == l1 for k in kws))
        pct = 100 * holders / n_unit_profs if n_unit_profs else 0.0
        l1_cov_rows.append({"parent_l1": l1, "n_faculty": holders, "coverage_pct": round(pct, 1),
                            "flag": "too_broad" if pct > 50 else "ok"})
    l1_cov = pd.DataFrame(l1_cov_rows).sort_values("n_faculty", ascending=False).reset_index(drop=True)
    too_broad = l1_cov.loc[l1_cov["flag"] == "too_broad", "parent_l1"].tolist()

    # ---- 3.4 虛無模型 ---------------------------------------------------
    null_df, null_samples, null_obs = run_null_model(kw_sets, n_null)
    null_df.insert(0, "unit", unit)

    # ---- 輸出 CSV / GraphML --------------------------------------------
    kw_cent.to_csv(out_dir / "keyword_centrality.csv", index=False, encoding="utf-8-sig")
    prof_cent.to_csv(out_dir / "professor_centrality.csv", index=False, encoding="utf-8-sig")
    kw_comm_df.to_csv(out_dir / "keyword_communities.csv", index=False, encoding="utf-8-sig")
    prof_comm_df.to_csv(out_dir / "professor_communities.csv", index=False, encoding="utf-8-sig")
    null_df.to_csv(out_dir / "null_model_test.csv", index=False, encoding="utf-8-sig")
    lam_df.to_csv(out_dir / "lambda_sensitivity.csv", index=False, encoding="utf-8-sig")
    l1_cov.to_csv(out_dir / "l1_coverage_guardrail.csv", index=False, encoding="utf-8-sig")

    Bx = B.copy()
    for n, d in Bx.nodes(data=True):
        d["label"] = str(n)
    nx.write_graphml(Bx, out_dir / "prof_keyword_network.graphml")
    KGx = KG.copy()
    for n in KGx.nodes():
        KGx.nodes[n]["community"] = int(kw_part.get(n, -1))
    nx.write_graphml(KGx, out_dir / "keyword_network.graphml")
    PGx = PG.copy()
    for n in PGx.nodes():
        PGx.nodes[n]["community"] = int(prof_part.get(n, -1))
    nx.write_graphml(PGx, out_dir / "professor_network.graphml")

    no_zh = [
        {"node_id": n, "name_en": B.nodes[n]["name_en"]}
        for n in profs_all
        if not CJK_RE.search(str(B.nodes[n]["professor_name"]))
    ]

    metrics = {
        "unit": unit,
        "level": level,
        "school_ids": meta["school_ids"],
        "deduplicated": meta["dedup"],
        "lambda": LAMBDA,
        "seed": SEED,
        "coverage": {
            "n_records": meta["n_records"],
            "n_entities": meta["n_entities"],
            "n_duplicate_records_removed": meta["n_duplicate_persons_removed"],
            "n_in_network": meta["n_in_network"],
            "n_excluded_no_keywords": meta["n_excluded_no_keywords"],
            "coverage_pct": meta["coverage_pct"],
        },
        "bipartite": {
            "n_professors": len(profs_all),
            "n_keywords": len(kws_all),
            "n_edges": B.number_of_edges(),
            "density": bip_density,
            "n_connected_components": n_components,
            "lcc_n_nodes": LCC.number_of_nodes(),
            "lcc_n_professors": len(lcc_profs),
            "lcc_n_keywords": len(lcc_kws),
            "lcc_node_share": LCC.number_of_nodes() / B.number_of_nodes() if B.number_of_nodes() else 0.0,
            "lcc_professor_share": len(lcc_profs) / len(profs_all) if profs_all else 0.0,
            "professor_degree_mean": float(np.mean(list(prof_deg.values()))) if prof_deg else 0.0,
            "professor_degree_max": int(max(prof_deg.values())) if prof_deg else 0,
            "keyword_degree_mean": float(np.mean(list(kw_deg.values()))) if kw_deg else 0.0,
            "keyword_degree_max": int(max(kw_deg.values())) if kw_deg else 0,
        },
        "keyword_network": {
            "n_nodes": KG.number_of_nodes(),
            "n_edges": KG.number_of_edges(),
            "density": nx.density(KG),
            "mean_degree": 2 * KG.number_of_edges() / KG.number_of_nodes() if KG.number_of_nodes() else 0.0,
            "mean_weight": float(np.mean([d["weight"] for _, _, d in KG.edges(data=True)])) if KG.number_of_edges() else 0.0,
            "avg_clustering": float(nx.average_clustering(KG)) if KG.number_of_nodes() > 2 else 0.0,
            "modularity_q_by_resolution": {str(r): kw_q[r] for r in RESOLUTIONS},
            "n_communities_by_resolution": {str(r): kw_ncomm[r] for r in RESOLUTIONS},
            "modularity_q": kw_q[1.0],
            "n_communities": kw_ncomm[1.0],
            "girvan_newman": kw_gn,
            "taxonomy_alignment_nmi": nmi_kw,
            "taxonomy_alignment_ari": ari_kw,
            "pearson_degree_betweenness_r": float(r_deg_btw),
            "pearson_degree_betweenness_p": float(p_deg_btw),
        },
        "professor_network": {
            "n_nodes": PG.number_of_nodes(),
            "n_edges": PG.number_of_edges(),
            "density": nx.density(PG),
            "mean_degree": 2 * PG.number_of_edges() / PG.number_of_nodes() if PG.number_of_nodes() else 0.0,
            "avg_clustering": float(nx.average_clustering(PG)) if PG.number_of_nodes() > 2 else 0.0,
            "modularity_q_by_resolution": {str(r): prof_q[r] for r in RESOLUTIONS},
            "n_communities_by_resolution": {str(r): prof_ncomm[r] for r in RESOLUTIONS},
            "modularity_q": prof_q[1.0],
            "n_communities": prof_ncomm[1.0],
            "assortativity_dominant_l1": assort_l1,
            "assortativity_role_groups": assort_role,
            "assortativity_school": assort_school,
            "community_vs_dominant_l1_nmi": nmi_prof_l1,
            "community_vs_dominant_l1_ari": ari_prof_l1,
        },
        "guardrails": {
            "l1_too_broad": too_broad,
            "lambda_over_merging": lam_df.loc[lam_df["flag_over_merging"], "lambda"].tolist(),
        },
        "fonts": {
            "cjk_font": CJK_FONT,
            "detection": CJK_FONT_NOTE,
            "fallback_to_name_en": not FONT_OK,
            "n_professors_without_chinese_name": len(no_zh),
            "professors_without_chinese_name": no_zh,
        },
    }
    with (out_dir / "network_metrics.json").open("w", encoding="utf-8") as fh:
        json.dump(metrics, fh, ensure_ascii=False, indent=2)

    ctx = dict(
        unit=unit, level=level, meta=meta, out_dir=out_dir, fig_dir=fig_dir,
        B=B, LCC=LCC, KG=KG, PG=PG, kw_sets=kw_sets, profs_all=profs_all, kws_all=kws_all,
        kw_cent=kw_cent, prof_cent=prof_cent, kw_part=kw_part, prof_part=prof_part,
        kw_df=kw_df, prof_df=prof_df, dominant_l1=dominant_l1, lam_df=lam_df,
        null_df=null_df, null_samples=null_samples, null_obs=null_obs,
        allowed_cjk=ALLOWED_CJK, metrics=metrics, l1_cov=l1_cov,
        r_deg_btw=r_deg_btw, p_deg_btw=p_deg_btw,
    )
    make_figures(ctx)
    return metrics

## 每個分析單位的圖表（01~17，組層級與全體另加 18~20）

In [10]:
import matplotlib.patches


def make_figures(ctx: dict) -> None:
    unit = ctx["unit"]
    level = ctx["level"]
    fig_dir = ctx["fig_dir"]
    B, LCC, KG, PG = ctx["B"], ctx["LCC"], ctx["KG"], ctx["PG"]
    kw_cent, prof_cent = ctx["kw_cent"], ctx["prof_cent"]
    kw_part, prof_part = ctx["kw_part"], ctx["prof_part"]
    allowed = ctx["allowed_cjk"]
    metrics = ctx["metrics"]

    def save(fig, name, caption=None):
        save_figure(fig, fig_dir / name, unit, allowed, caption)

    def kw_label(n):
        return n

    def p_label(n):
        return prof_label(n, B.nodes[n])

    # ---------- 01 bipartite network ----------------------------------
    fig, ax = plt.subplots(figsize=(16, 13))
    pos_b = nx.spring_layout(B, seed=SEED, iterations=200,
                         k=1.6 / math.sqrt(max(B.number_of_nodes(), 1)))
    profs = [n for n, d in B.nodes(data=True) if d["bipartite"] == 0]
    kws = [n for n, d in B.nodes(data=True) if d["bipartite"] == 1]
    nx.draw_networkx_edges(ax=ax, G=B, pos=pos_b, width=0.4, alpha=0.2, edge_color="#999999")
    nx.draw_networkx_nodes(ax=ax, G=B, pos=pos_b, nodelist=profs, node_size=45,
                           node_color=COLOR_PROF, linewidths=0.3, edgecolors="white", label="Professor")
    kw_sizes = scaled_sizes(dict(B.degree(kws)), kws, 60, 700)
    nx.draw_networkx_nodes(ax=ax, G=B, pos=pos_b, nodelist=kws, node_size=kw_sizes,
                           node_color=COLOR_KW, linewidths=0.3, edgecolors="white", label="Research field")
    ax.set_title(f"{unit} — Professor-Research Field bipartite network")
    ax.legend(loc="upper right", frameon=True)
    ax.set_axis_off()
    size_map = {**{u: 45 for u in profs}, **dict(zip(kws, kw_sizes))}
    label_map = {u: u for u in top_label_set(dict(B.degree(kws)), kws, 15)}
    label_colors = {u: darken(COLOR_KW) for u in label_map}
    if B.number_of_nodes() <= 80:
        label_map.update({u: p_label(u) for u in profs})
        label_colors.update({u: darken(COLOR_PROF) for u in profs})
    annotate_without_overlap(ax, pos_b, label_map, size_map, font_size=7.5,
                             obstacle_nodes=list(B.nodes()), label_colors=label_colors)
    save(fig, "01_bipartite_network.png", caption_for(B, False, None))

    # ---------- 02 largest connected component -------------------------
    fig, ax = plt.subplots(figsize=(16, 13))
    pos_l = nx.spring_layout(LCC, seed=SEED, iterations=200,
                         k=1.6 / math.sqrt(max(LCC.number_of_nodes(), 1)))
    lprofs = [n for n, d in LCC.nodes(data=True) if d["bipartite"] == 0]
    lkws = [n for n, d in LCC.nodes(data=True) if d["bipartite"] == 1]
    nx.draw_networkx_edges(ax=ax, G=LCC, pos=pos_l, width=0.4, alpha=0.2, edge_color="#999999")
    nx.draw_networkx_nodes(ax=ax, G=LCC, pos=pos_l, nodelist=lprofs, node_size=45,
                           node_color=COLOR_PROF, linewidths=0.3, edgecolors="white", label="Professor")
    lkw_sizes = scaled_sizes(dict(LCC.degree(lkws)), lkws, 60, 700)
    nx.draw_networkx_nodes(ax=ax, G=LCC, pos=pos_l, nodelist=lkws, node_size=lkw_sizes,
                           node_color=COLOR_KW, linewidths=0.3, edgecolors="white", label="Research field")
    b = metrics["bipartite"]
    ax.set_title(
        f"{unit} — Largest connected component "
        f"({b['lcc_n_nodes']}/{b['n_professors'] + b['n_keywords']} nodes = {100 * b['lcc_node_share']:.1f}%; "
        f"{100 * b['lcc_professor_share']:.1f}% of professors)"
    )
    ax.legend(loc="upper right", frameon=True)
    ax.set_axis_off()
    _lcc_labels = {u: u for u in top_label_set(dict(LCC.degree(lkws)), lkws, 15)}
    annotate_without_overlap(
        ax, pos_l, _lcc_labels,
        {**{u: 45 for u in lprofs}, **dict(zip(lkws, lkw_sizes))},
        font_size=7.5, obstacle_nodes=list(LCC.nodes()),
        label_colors={u: darken(COLOR_KW) for u in _lcc_labels},
    )
    save(fig, "02_largest_component.png", caption_for(LCC, False, None))

    # ---------- keyword network layouts --------------------------------
    KGb, KG_BB = backbone_for_plot(KG)
    pos_k = nx.spring_layout(KG, seed=SEED, weight="cosine", iterations=200,
                         k=2.2 / math.sqrt(max(KG.number_of_nodes(), 1)))
    knodes = list(KGb.nodes())
    deg_c = dict(zip(kw_cent["node"], kw_cent["degree_centrality"]))
    btw_c = dict(zip(kw_cent["node"], kw_cent["betweenness"]))

    for name, values, title in [
        ("03_keyword_network_degree.png", deg_c, "node size = degree centrality"),
        ("04_keyword_network_betweenness.png", btw_c, "node size = betweenness centrality"),
    ]:
        fig, ax = plt.subplots(figsize=(15, 12))
        draw_network(ax, KGb, pos_k, [COLOR_KW] * len(knodes), values,
                     top_label_set(values, knodes, 15), kw_label, font_size=8)
        ax.set_title(f"{unit} — Research field co-occurrence network ({title})")
        save(fig, name, caption_for(KGb, KG_BB, None))

    # ---------- 05 keyword communities ---------------------------------
    fig, ax = plt.subplots(figsize=(15, 12))
    comm_ids = sorted({kw_part.get(n, -1) for n in knodes})
    ccolor = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(comm_ids)}
    draw_network(ax, KGb, pos_k, [ccolor[kw_part.get(n, -1)] for n in knodes], deg_c,
                 top_label_set(deg_c, knodes, 15), kw_label, font_size=8,
                 label_colors={n: darken(ccolor[kw_part.get(n, -1)]) for n in knodes})
    q = metrics["keyword_network"]["modularity_q"]
    ax.set_title(f"{unit} — Research field communities (Louvain, resolution 1.0, Q = {q:.3f})")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=ccolor[c], label=f"Community {c}") for c in comm_ids],
              loc="upper right", frameon=True)
    save(fig, "05_keyword_network_communities.png", caption_for(KGb, KG_BB, None))

    # ---------- 06 keyword network by taxonomy -------------------------
    fig, ax = plt.subplots(figsize=(15, 12))
    l1s = sorted({KW2L1[n] for n in knodes})
    draw_network(ax, KGb, pos_k, [L1_COLOR[KW2L1[n]] for n in knodes], deg_c,
                 top_label_set(deg_c, knodes, 15), kw_label, font_size=8,
                 label_colors={n: darken(L1_COLOR[KW2L1[n]]) for n in knodes})
    nmi = metrics["keyword_network"]["taxonomy_alignment_nmi"]
    ari = metrics["keyword_network"]["taxonomy_alignment_ari"]
    ax.set_title(f"{unit} — Research field network coloured by taxonomy L1 (NMI = {nmi:.3f}, ARI = {ari:.3f})")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=L1_COLOR[l], label=l) for l in l1s],
              loc="upper right", frameon=True, fontsize=8)
    save(fig, "06_keyword_network_by_taxonomy.png", caption_for(KGb, KG_BB, None))

    # ---------- professor network layouts ------------------------------
    PGb, PG_BB = backbone_for_plot(PG)
    pos_p = nx.spring_layout(PG, seed=SEED, weight="w_cos", iterations=200,
                         k=2.2 / math.sqrt(max(PG.number_of_nodes(), 1)))
    pnodes = list(PGb.nodes())
    pdeg = dict(zip(prof_cent["node"], prof_cent["degree_centrality"]))
    pbtw = dict(zip(prof_cent["node"], prof_cent["betweenness"]))

    for name, values, title in [
        ("07_professor_network_degree.png", pdeg, "node size = degree centrality"),
        ("08_professor_network_betweenness.png", pbtw, "node size = betweenness centrality"),
    ]:
        fig, ax = plt.subplots(figsize=(15, 12))
        draw_network(ax, PGb, pos_p, [COLOR_PROF] * len(pnodes), values,
                     top_label_set(values, pnodes, 15), p_label, font_size=7.5)
        ax.set_title(f"{unit} — Professor co-research network ({title})")
        save(fig, name, caption_for(PGb, PG_BB, LAMBDA))

    # ---------- 09 professor communities -------------------------------
    fig, ax = plt.subplots(figsize=(15, 12))
    pcomm_ids = sorted({prof_part.get(n, -1) for n in pnodes})
    pcolor = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(pcomm_ids)}
    draw_network(ax, PGb, pos_p, [pcolor[prof_part.get(n, -1)] for n in pnodes], pdeg,
                 top_label_set(pdeg, pnodes, 15), p_label, font_size=7.5,
                 label_colors={n: darken(pcolor[prof_part.get(n, -1)]) for n in pnodes})
    pq = metrics["professor_network"]["modularity_q"]
    ax.set_title(f"{unit} — Professor communities (Louvain, resolution 1.0, Q = {pq:.3f})")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=pcolor[c], label=f"Community {c}") for c in pcomm_ids],
              loc="upper right", frameon=True)
    save(fig, "09_professor_network_communities.png", caption_for(PGb, PG_BB, LAMBDA))

    # ---------- 10 top keywords bar ------------------------------------
    top_kw = kw_cent.nlargest(20, "n_faculty_in_unit").sort_values("n_faculty_in_unit")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top_kw["node"], top_kw["n_faculty_in_unit"],
            color=[L1_COLOR[KW2L1[k]] for k in top_kw["node"]])
    for y, v in enumerate(top_kw["n_faculty_in_unit"]):
        ax.text(v + 0.1, y, str(int(v)), va="center", fontsize=8)
    ax.set_xlabel("Number of professors")
    ax.set_ylabel("Research field")
    ax.set_title(f"{unit} — Top {len(top_kw)} research fields by professor count")
    ax.legend(handles=[Line2D([], [], marker="s", ls="", color=L1_COLOR[l], label=l)
                       for l in sorted({KW2L1[k] for k in top_kw["node"]})],
              loc="lower right", fontsize=7, frameon=True)
    fig.tight_layout()
    save(fig, "10_top_keywords_bar.png")

    # ---------- 11 degree distribution ---------------------------------
    profs_all, kws_all = ctx["profs_all"], ctx["kws_all"]
    pdg = np.array([d for _, d in B.degree(profs_all)])
    kdg = np.array([d for _, d in B.degree(kws_all)])
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for j, (vals, label, color) in enumerate(
        [(pdg, "Professor degree (number of research fields)", COLOR_PROF),
         (kdg, "Research field degree (number of professors)", COLOR_KW)]
    ):
        axes[0, j].hist(vals, bins=min(25, max(3, len(set(vals.tolist())))), color=color, edgecolor="white")
        axes[0, j].set_xlabel(label)
        axes[0, j].set_ylabel("Count")
        axes[0, j].set_title("Linear scale")
        cnt = Counter(vals.tolist())
        xs = np.array(sorted(cnt))
        ys = np.array([cnt[x] for x in xs], dtype=float)
        m = xs > 0
        axes[1, j].loglog(xs[m], ys[m], "o", color=color, markersize=5)
        axes[1, j].set_xlabel(label + " (log)")
        axes[1, j].set_ylabel("Count (log)")
        axes[1, j].set_title("Log-log scale")
    fig.suptitle(f"{unit} — Degree distributions of the bipartite network")
    fig.tight_layout()
    save(fig, "11_degree_distribution.png")

    # ---------- 12 centrality scatter ----------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, df, color, title, rp in [
        (axes[0], kw_cent, COLOR_KW, "Research field network", (ctx["r_deg_btw"], ctx["p_deg_btw"])),
        (axes[1], prof_cent, COLOR_PROF, "Professor network", None),
    ]:
        if rp is None:
            rp = pearsonr(df["degree"], df["betweenness"]) if len(df) > 2 else (float("nan"), float("nan"))
        ax.scatter(df["degree"], df["betweenness"], s=22, color=color, alpha=0.75,
                   edgecolors="white", linewidths=0.4)
        ax.set_xlabel("Degree")
        ax.set_ylabel("Betweenness centrality")
        pv = rp[1]
        pv_txt = "p < 0.001" if pv == pv and pv < 0.001 else f"p = {pv:.3f}"
        ax.set_title(f"{title}\nPearson r = {rp[0]:.3f}, {pv_txt}")
    fig.suptitle(f"{unit} — Degree vs betweenness centrality")
    fig.tight_layout()
    save(fig, "12_centrality_scatter.png")

    # ---------- 13 co-occurrence heatmap -------------------------------
    top25 = list(kw_cent.nlargest(25, "n_faculty_in_unit")["node"])
    mat = np.zeros((len(top25), len(top25)))
    for i, a in enumerate(top25):
        for j, bkw in enumerate(top25):
            if i != j and KG.has_edge(a, bkw):
                mat[i, j] = KG[a][bkw]["weight"]
    fig, ax = plt.subplots(figsize=(11, 9.5))
    im = ax.imshow(mat, cmap="YlOrRd")
    ax.set_xticks(range(len(top25)))
    ax.set_xticklabels(top25, rotation=90, fontsize=7)
    ax.set_yticks(range(len(top25)))
    ax.set_yticklabels(top25, fontsize=7)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Number of shared professors")
    ax.set_title(f"{unit} — Co-occurrence matrix of top {len(top25)} research fields")
    fig.tight_layout()
    save(fig, "13_keyword_cooccurrence_heatmap.png")

    # ---------- 14 community composition -------------------------------
    comp = pd.DataFrame({"node": list(kw_part), "community": [kw_part[n] for n in kw_part]})
    comp["parent_l1"] = comp["node"].map(KW2L1)
    pivot = comp.pivot_table(index="community", columns="parent_l1", values="node", aggfunc="count").fillna(0)
    fig, ax = plt.subplots(figsize=(12, 6.5))
    bottom = np.zeros(len(pivot))
    for l1 in pivot.columns:
        ax.bar(pivot.index.astype(str), pivot[l1], bottom=bottom, color=L1_COLOR[l1], label=l1)
        bottom += pivot[l1].values
    ax.set_xlabel("Louvain community")
    ax.set_ylabel("Number of research fields")
    ax.set_title(f"{unit} — Taxonomy L1 composition of detected research field communities")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    fig.tight_layout()
    save(fig, "14_community_composition.png")

    # ---------- 15 null model comparison -------------------------------
    null_df, samples = ctx["null_df"], ctx["null_samples"]
    labels = {
        "keyword_density": "Research field network density",
        "keyword_modularity_q": "Research field modularity Q",
        "keyword_mean_cooccurrence": "Mean co-occurrence count",
        "professor_clustering": "Professor network clustering coefficient",
    }
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, (metric, label) in zip(axes.ravel(), labels.items()):
        row = null_df[null_df["metric"] == metric].iloc[0]
        arr = samples[metric]
        ax.hist(arr, bins=35, color="#B0B7C3", edgecolor="white", label="Null model (1,000 rewirings)")
        ax.axvline(row["observed"], color="#C44E52", lw=2, label="Observed")
        ax.set_xlabel(label)
        ax.set_ylabel("Frequency")
        pv = row["p_empirical_two_sided"]
        pv_txt = "p < 0.002" if pv <= 0.002 else f"p = {pv:.3f}"
        ax.set_title(f"z = {row['z_score']:.2f}, {pv_txt}")
        ax.legend(fontsize=8)
    fig.suptitle(f"{unit} — Observed statistics vs degree-preserving bipartite null model")
    fig.tight_layout()
    save(fig, "15_null_model_comparison.png")

    # ---------- 16 lambda sensitivity ----------------------------------
    lam_df = ctx["lam_df"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, col, label in [
        (axes[0], "density", "Professor network density"),
        (axes[1], "modularity_q", "Modularity Q"),
        (axes[2], "lcc_share", "Largest component share"),
    ]:
        ax.plot(lam_df["lambda"], lam_df[col], "o-", color=COLOR_PROF)
        ax.axvline(LAMBDA, color="#C44E52", ls="--", lw=1, label=f"default lambda = {LAMBDA}")
        ax.set_xlabel("lambda")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend(fontsize=8)
    fig.suptitle(f"{unit} — Sensitivity of the professor network to the parent-discount weight lambda")
    fig.tight_layout()
    save(fig, "16_lambda_sensitivity.png")

    # ---------- 17 taxonomy sunburst -----------------------------------
    kw_unit = ctx["kw_df"].set_index("keyword_en")["n_faculty_in_unit"].to_dict()
    l1_tot: Counter = Counter()
    l2_tot: Counter = Counter()
    for k, v in kw_unit.items():
        l1_tot[KW2L1[k]] += v
        l2_tot[(KW2L1[k], KW2L2[k])] += v
    l1_keys = sorted(l1_tot, key=lambda x: -l1_tot[x])
    l2_keys = sorted(l2_tot, key=lambda x: (l1_keys.index(x[0]), -l2_tot[x]))
    l3_keys = sorted(kw_unit, key=lambda k: (l1_keys.index(KW2L1[k]),
                                             l2_keys.index((KW2L1[k], KW2L2[k])), -kw_unit[k]))
    total = sum(l1_tot.values()) or 1

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_aspect("equal")
    ax.axis("off")
    rings = [
        (0.55, 0.25, l1_keys, [l1_tot[k] for k in l1_keys], [L1_COLOR[k] for k in l1_keys],
         lambda k: k, 8, 0.95),
        (0.80, 0.20, l2_keys, [l2_tot[k] for k in l2_keys], [L1_COLOR[k[0]] for k in l2_keys],
         lambda k: k[1], 7, 0.8),
        (0.98, 0.16, l3_keys, [kw_unit[k] for k in l3_keys], [L1_COLOR[KW2L1[k]] for k in l3_keys],
         lambda k: k, 6, 0.6),
    ]
    for radius, width, keys, sizes, colors, labeler, fs, alpha in rings:
        start = 90.0
        for key, size, color in zip(keys, sizes, colors):
            extent = 360.0 * size / total
            mid = start - extent / 2
            ax.add_patch(matplotlib.patches.Wedge((0, 0), radius, start - extent, start,
                                                  width=width, facecolor=color, edgecolor="white",
                                                  lw=0.6, alpha=alpha))
            if extent >= 10:
                rad = math.radians(mid)
                r = radius - width / 2
                ax.text(r * math.cos(rad), r * math.sin(rad), labeler(key),
                        ha="center", va="center", fontsize=fs)
            start -= extent
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(f"{unit} — Research taxonomy L1 -> L2 -> L3 distribution (ring angle = professor mentions)")
    save(fig, "17_taxonomy_sunburst.png")

    # ---------- 21~28 word clouds -------------------------------------
    # 字級與中心性成正比；社群版本把同一個 Louvain 社群的詞排在同一塊、共用顏色。
    kw_comm_color = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(sorted(set(kw_part.values())))}
    prof_comm_color = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(sorted(set(prof_part.values())))}

    wc_specs = [
        ("21", "keyword", "degree_centrality", "degree centrality", False),
        ("22", "keyword", "betweenness", "betweenness centrality", False),
        ("23", "professor", "degree_centrality", "degree centrality", False),
        ("24", "professor", "betweenness", "betweenness centrality", False),
        ("25", "keyword", "degree_centrality", "degree centrality", True),
        ("26", "keyword", "betweenness", "betweenness centrality", True),
        ("27", "professor", "degree_centrality", "degree centrality", True),
        ("28", "professor", "betweenness", "betweenness centrality", True),
    ]
    wc_names = {
        ("keyword", "degree_centrality", False): "21_keyword_wordcloud_degree.png",
        ("keyword", "betweenness", False): "22_keyword_wordcloud_betweenness.png",
        ("professor", "degree_centrality", False): "23_professor_wordcloud_degree.png",
        ("professor", "betweenness", False): "24_professor_wordcloud_betweenness.png",
        ("keyword", "degree_centrality", True): "25_keyword_wordcloud_degree_by_community.png",
        ("keyword", "betweenness", True): "26_keyword_wordcloud_betweenness_by_community.png",
        ("professor", "degree_centrality", True): "27_professor_wordcloud_degree_by_community.png",
        ("professor", "betweenness", True): "28_professor_wordcloud_betweenness_by_community.png",
    }

    for _, side, col, metric_label, by_comm in wc_specs:
        if side == "keyword":
            df, part, ccolor = kw_cent, kw_part, kw_comm_color
            text_of, kind_label = (lambda n: n), "Research field"
        else:
            df, part, ccolor = prof_cent, prof_part, prof_comm_color
            text_of, kind_label = p_label, "Professor"

        entries = [
            {
                "key": row.node,
                "text": text_of(row.node),
                "value": float(getattr(row, col)),
                "color": darken(ccolor[part.get(row.node, -1)]) if by_comm else "#333333",
            }
            for row in df.itertuples(index=False)
        ]
        fig, ax = plt.subplots(figsize=(12, 9))  # draw_wordcloud 會依內容重設尺寸
        stats = draw_wordcloud(
            ax, entries,
            cluster_of=part if by_comm else None,
            cluster_colors=ccolor if by_comm else None,
        )
        suffix = " grouped by Louvain community" if by_comm else ""
        ax.set_title(f"{unit} — {kind_label} word cloud, size proportional to {metric_label}{suffix}")
        save(fig, wc_names[(side, col, by_comm)],
             wordcloud_caption(stats, metric_label, len(entries),
                               LAMBDA if side == "professor" else None))

    if level == "school":
        return

    # ---------- 18 professor network by school -------------------------
    school_of = {n: str(B.nodes[n]["school_ids"]).split(" | ") for n in PG.nodes()}
    fig, ax = plt.subplots(figsize=(15, 12))
    colors = [SCHOOL_COLOR[school_of[n][0]] for n in pnodes]
    draw_network(ax, PGb, pos_p, colors, pdeg, top_label_set(pdeg, pnodes, 15), p_label, font_size=7.5,
                 label_colors={n: darken(SCHOOL_COLOR[school_of[n][0]]) for n in pnodes})
    used = sorted({s for n in pnodes for s in school_of[n]})
    ax.set_title(f"{unit} — Professor network coloured by school")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=SCHOOL_COLOR[s], label=s) for s in used],
              loc="upper right", frameon=True, fontsize=8)
    save(fig, "18_professor_network_by_school.png", caption_for(PGb, PG_BB, LAMBDA))

    # ---------- 19 cross-school edges heatmap --------------------------
    schools = sorted({s for n in PG.nodes() for s in school_of[n]})
    idx = {s: i for i, s in enumerate(schools)}
    mat = np.zeros((len(schools), len(schools)))
    for u, v, d in PG.edges(data=True):
        for su in school_of[u]:
            for sv in school_of[v]:
                mat[idx[su], idx[sv]] += d["weight"]
                if su != sv:
                    mat[idx[sv], idx[su]] += d["weight"]
    fig, ax = plt.subplots(figsize=(9.5, 8))
    im = ax.imshow(mat, cmap="Blues")
    ax.set_xticks(range(len(schools)))
    ax.set_xticklabels(schools, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(schools)))
    ax.set_yticklabels(schools, fontsize=8)
    for i in range(len(schools)):
        for j in range(len(schools)):
            ax.text(j, i, f"{mat[i, j]:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if mat[i, j] > mat.max() * 0.6 else "#333333")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Sum of professor-network edge weights")
    ax.set_title(f"{unit} — School x school connection strength")
    fig.tight_layout()
    save(fig, "19_cross_school_edges.png")

    # ---------- 20 joint appointment highlight -------------------------
    fig, ax = plt.subplots(figsize=(15, 12))
    joint = {n for n in pnodes if B.nodes[n]["n_appointments"] > 1}
    nx.draw_networkx_edges(ax=ax, G=PGb, pos=pos_p, width=edge_widths(PGb), alpha=0.15, edge_color="#999999")
    joint_sizes = {n: (260 if n in joint else 40) for n in pnodes}
    nx.draw_networkx_nodes(ax=ax, G=PGb, pos=pos_p, nodelist=pnodes,
                           node_size=[joint_sizes[n] for n in pnodes],
                           node_color=["#C44E52" if n in joint else "#C9CFD8" for n in pnodes],
                           linewidths=0.4, edgecolors="white")
    ax.set_axis_off()
    annotate_without_overlap(ax, pos_p, {n: p_label(n) for n in joint}, joint_sizes,
                             font_size=7.5, obstacle_nodes=pnodes,
                             label_colors={n: darken("#C44E52", 0.8) for n in joint})
    ax.set_title(f"{unit} — Professors with joint appointments ({len(joint)} of {len(pnodes)} nodes)")
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color="#C44E52",
                              label="Joint appointment (n_appointments > 1)"),
                       Line2D([], [], marker="o", ls="", color="#C9CFD8", label="Single appointment")],
              loc="upper right", frameon=True, fontsize=8)
    save(fig, "20_joint_appointment_highlight.png", caption_for(PGb, PG_BB, LAMBDA))

## 執行 18 個分析單位

虛無模型（18 單位 × 1,000 次隨機化）為主要耗時來源。

In [11]:
ALL_METRICS = []
t_all = time.time()
for meta in MANIFEST:
    t0 = time.time()
    m = analyse_unit(meta)
    ALL_METRICS.append(m)
    print(
        f"{meta['unit']:28s} {meta['level']:8s} "
        f"prof={m['bipartite']['n_professors']:4d} kw={m['bipartite']['n_keywords']:4d} "
        f"KG(n={m['keyword_network']['n_nodes']:3d},e={m['keyword_network']['n_edges']:5d},Q={m['keyword_network']['modularity_q']:.3f}) "
        f"PG(n={m['professor_network']['n_nodes']:4d},e={m['professor_network']['n_edges']:6d},Q={m['professor_network']['modularity_q']:.3f}) "
        f"[{time.time() - t0:6.1f}s]"
    )
print(f"\ntotal elapsed: {time.time() - t_all:.1f}s")

nccu-fin                     school   prof=  23 kw=  24 KG(n= 22,e=   51,Q=0.276) PG(n=  21,e=   159,Q=0.123) [  18.0s]


nccu-mb                      school   prof=  18 kw=  23 KG(n= 19,e=   41,Q=0.289) PG(n=  16,e=    49,Q=0.226) [  12.6s]


ncku-cs-ie                   school   prof=  51 kw=  66 KG(n= 65,e=  286,Q=0.397) PG(n=  50,e=   493,Q=0.235) [  30.6s]


ncku-stats-ds                school   prof=  39 kw=  44 KG(n= 44,e=  169,Q=0.466) PG(n=  39,e=   345,Q=0.171) [  22.3s]


nthu-cs                      school   prof=  76 kw=  65 KG(n= 63,e=  216,Q=0.422) PG(n=  74,e=   606,Q=0.439) [  30.8s]


<site-packages>\networkx\algorithms\assortativity\correlation.py:282: RuntimeWarning: invalid value encountered in scalar divide
  r = (t - s) / (1 - s)


... (9 outputs left out)

fin-and-econ                 group    prof= 151 kw=  69 KG(n= 68,e=  323,Q=0.347) PG(n= 148,e=  3720,Q=0.304) [  42.8s]


ds                           group    prof= 114 kw=  86 KG(n= 79,e=  382,Q=0.397) PG(n= 107,e=  1555,Q=0.414) [  40.3s]


cs                           group    prof= 381 kw= 115 KG(n=113,e=  851,Q=0.295) PG(n= 379,e= 18688,Q=0.315) [ 102.1s]


overall                      overall  prof= 623 kw= 170 KG(n=164,e= 1276,Q=0.383) PG(n= 616,e= 31823,Q=0.396) [ 183.7s]

total elapsed: 684.3s


## 語言檢查與圖檔盤點

In [12]:
lang_df = pd.DataFrame(LANG_CHECK_ROWS)
lang_df.to_csv(OUT_ROOT / "figure_language_check.csv", index=False, encoding="utf-8-sig")
fig_df = pd.DataFrame(FIGURE_INDEX)
fig_df.to_csv(OUT_ROOT / "figure_index.csv", index=False, encoding="utf-8-sig")

violations = lang_df[lang_df["n_violations"] > 0]
print(f"figures generated      : {len(fig_df)}")
print(f"text elements scanned  : {int(lang_df['n_text_elements'].sum())}")
print(f"CJK text elements      : {int(lang_df['n_cjk_elements'].sum())}  (professor names only)")
print(f"language violations    : {len(violations)}")
if len(violations):
    print(violations.head(20).to_string(index=False))

counts = fig_df.groupby("unit").size()
expected = {m["unit"]: (25 if m["level"] == "school" else 28) for m in MANIFEST}
bad_counts = {u: (int(counts.get(u, 0)), e) for u, e in expected.items() if int(counts.get(u, 0)) != e}
print(f"\n每單位圖檔數量正確: {not bad_counts}" + (f"  {bad_counts}" if bad_counts else ""))

figures generated      : 462
text elements scanned  : 27247
CJK text elements      : 6392  (professor names only)
language violations    : 0

每單位圖檔數量正確: True


## 單位摘要與驗證

In [13]:
summary = pd.DataFrame(
    [
        {
            "unit": m["unit"],
            "level": m["level"],
            "n_professors": m["bipartite"]["n_professors"],
            "n_excluded": m["coverage"]["n_excluded_no_keywords"],
            "coverage_pct": m["coverage"]["coverage_pct"],
            "n_keywords": m["bipartite"]["n_keywords"],
            "bipartite_density": m["bipartite"]["density"],
            "lcc_prof_share": m["bipartite"]["lcc_professor_share"],
            "kw_density": m["keyword_network"]["density"],
            "kw_mean_degree": m["keyword_network"]["mean_degree"],
            "kw_modularity_q": m["keyword_network"]["modularity_q"],
            "kw_n_communities": m["keyword_network"]["n_communities"],
            "kw_taxonomy_nmi": m["keyword_network"]["taxonomy_alignment_nmi"],
            "prof_density": m["professor_network"]["density"],
            "prof_mean_degree": m["professor_network"]["mean_degree"],
            "prof_modularity_q": m["professor_network"]["modularity_q"],
            "prof_avg_clustering": m["professor_network"]["avg_clustering"],
            "l1_too_broad": " | ".join(m["guardrails"]["l1_too_broad"]),
        }
        for m in ALL_METRICS
    ]
)
pd.set_option("display.width", 250)
print(summary.round(4).to_string(index=False))
summary.to_csv(OUT_ROOT / "unit_summary.csv", index=False, encoding="utf-8-sig")

                     unit   level  n_professors  n_excluded  coverage_pct  n_keywords  bipartite_density  lcc_prof_share  kw_density  kw_mean_degree  kw_modularity_q  kw_n_communities  kw_taxonomy_nmi  prof_density  prof_mean_degree  prof_modularity_q  prof_avg_clustering                                                       l1_too_broad
                 nccu-fin  school            23           0         100.0          24             0.1123          0.9130      0.2208          4.6364           0.2759                 4           0.3527        0.7571           15.1429             0.1227               0.8924 Corporate Finance & Governance | Asset Pricing & Financial Markets
                  nccu-mb  school            18           4          81.8          23             0.1063          0.8889      0.2398          4.3158           0.2886                 4           0.4996        0.4083            6.1250             0.2264               0.7932                                       Financial

In [14]:
checks = []

# GraphML 可被 networkx 重新讀入且節點數相符
bad = []
for m in ALL_METRICS:
    d = OUT_ROOT / m["unit"]
    for fname, expected_n in [
        ("prof_keyword_network.graphml", m["bipartite"]["n_professors"] + m["bipartite"]["n_keywords"]),
        ("keyword_network.graphml", m["keyword_network"]["n_nodes"]),
        ("professor_network.graphml", m["professor_network"]["n_nodes"]),
    ]:
        g = nx.read_graphml(d / fname)
        if g.number_of_nodes() != expected_n:
            bad.append((m["unit"], fname, g.number_of_nodes(), expected_n))
checks.append(("GraphML 可被 networkx 重新讀入且節點數相符", not bad, str(bad[:5])))

# λ=0 與純 IDF 加權投影一致
sample_meta = MANIFEST[0]
sdir = ROOT / sample_meta["out_dir"]
sprof = pd.read_csv(sdir / "professor_nodes.csv", encoding="utf-8-sig")
sedge = pd.read_csv(sdir / "prof_keyword_edges.csv", encoding="utf-8-sig")
sB = build_bipartite(sprof, sedge)
slcc = sB.subgraph(max(nx.connected_components(sB), key=len))
skw = {p: set(slcc.neighbors(p)) for p in slcc if slcc.nodes[p]["bipartite"] == 0}
G0 = professor_projection(skw, 0.0)
ref = nx.Graph()
ref.add_nodes_from(skw)
for a in skw:
    for b in skw:
        if a < b:
            shared = skw[a] & skw[b]
            if shared:
                ref.add_edge(a, b, weight=sum(IDF[k] for k in shared))
same = set(G0.edges()) == set(ref.edges()) and all(
    abs(G0[u][v]["weight"] - ref[u][v]["weight"]) < 1e-9 for u, v in ref.edges()
)
checks.append((f"λ=0 的結果與純 IDF 加權投影一致（{sample_meta['unit']}）", same, ""))

# 每個單位的 network_metrics.json 記錄排除人數
bad = [m["unit"] for m in ALL_METRICS if "n_excluded_no_keywords" not in m["coverage"]]
checks.append(("每個單位的 network_metrics.json 記錄排除人數", not bad, str(bad)))

# 圖檔位置：data/ 內無任何圖檔
stray = list((ROOT / "data").rglob("*.png")) + list((ROOT / "data").rglob("*.svg"))
checks.append(("data/ 內無任何圖檔", not stray, str(stray[:5])))

# 圖檔數量
checks.append(("每單位圖檔數量正確（系所 25、組與全體 28）", not bad_counts, str(bad_counts)))

# 語言檢查
checks.append(("圖中除教授姓名外無中文字元", len(violations) == 0, str(violations["figure"].tolist()[:5])))

# 中文字型
checks.append((f"中文字型偵測成功（{CJK_FONT}）", FONT_OK, CJK_FONT_NOTE))

# 無中文名教授已記錄
n_no_zh = sum(m["fonts"]["n_professors_without_chinese_name"] for m in ALL_METRICS if m["level"] == "school")
checks.append((f"無中文名的教授已記錄於 network_metrics.json（系所層級合計 {n_no_zh} 位）", n_no_zh > 0, ""))

for label, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}" + (f"  {detail}" if not ok else ""))

assert all(ok for _, ok, _ in checks), "Stage 3 驗證未通過"
print("\nStage 3 完成。")

[PASS] GraphML 可被 networkx 重新讀入且節點數相符
[PASS] λ=0 的結果與純 IDF 加權投影一致（nccu-fin）
[PASS] 每個單位的 network_metrics.json 記錄排除人數
[PASS] data/ 內無任何圖檔
[PASS] 每單位圖檔數量正確（系所 25、組與全體 28）
[PASS] 圖中除教授姓名外無中文字元
[PASS] 中文字型偵測成功（Microsoft JhengHei）
[PASS] 無中文名的教授已記錄於 network_metrics.json（系所層級合計 13 位）

Stage 3 完成。
